In [ ]:
import os
import gc
import numpy as np
import torch
import whisper
import jiwer
import pandas as pd
from datasets import load_dataset, Audio
from transformers.models.whisper.english_normalizer import BasicTextNormalizer  # kept from earlier, independent of HF model
from tqdm.auto import tqdm
from pathlib import Path

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
LANG       = "ar_eg"          # FLEURS code; try "en_us", "fr_fr", "de_de", ...
SPLIT      = "test"
MODEL_SIZES = ["tiny", "small"]   # openai-whisper model sizes: tiny, base, small, medium, large, turbo
MAX_SAMPLES = None            # e.g. 50 for a quick smoke test, None for full split

# ------------------------------------------------------------------
# CPU tuning — physical cores, single inter-op thread
# ------------------------------------------------------------------
n_threads = max(1, os.cpu_count() // 2 or 1)
torch.set_num_threads(n_threads)
torch.set_num_interop_threads(1)
print(f"PyTorch intra-op threads: {torch.get_num_threads()}")

DEVICE = "cpu"   # openai-whisper respects this; it will not try to use CUDA
print(f"Device: {DEVICE}")

In [17]:
MODEL_DIR = Path("../models/baselines").resolve()
MODEL_DIR.mkdir(exist_ok=True)

for size in MODEL_SIZES:
    pt = MODEL_DIR / f"whisper_{size}.pt"
    if pt.exists():
        print(f"✓ {pt.name:<20s}  {pt.stat().st_size / 1e6:7.1f} MB  (already saved)")
        continue
    print(f"↓ one-time download of whisper-{size} …")
    m = whisper.load_model(size, device="cpu")
    # Same format whisper uses internally — dims + state_dict
    torch.save({"dims": m.dims, "model_state_dict": m.state_dict()}, pt)
    print(f"✓ {pt.name:<20s}  {pt.stat().st_size / 1e6:7.1f} MB  saved")
    del m

↓ one-time download of whisper-tiny …
✓ whisper_tiny.pt         151.1 MB  saved
↓ one-time download of whisper-small …
✓ whisper_small.pt        967.1 MB  saved


In [3]:
dataset = load_dataset("google/fleurs", LANG, split=SPLIT)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))  # explicit 16 kHz

if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f"Loaded {len(dataset)} utterances from {LANG}/{SPLIT}.")
print("Example transcription:", dataset[0]["transcription"][:120], "...")

Loaded 428 utterances from ar_eg/test.
Example transcription: تشكلت في المحيط الأطلسي اليوم عاشر عاصفة مُسماة لموسم الأعاصير الأطلسية العاصفة شبه الاستوائية جيري ...


In [10]:
# Whisper's official normaliser: lower‑cases, strips punctuation, collapses whitespace.
normalizer = BasicTextNormalizer()

In [4]:
def collect_valid_indices(ds) -> list[int]:
    valid, corrupt = [], []
    for i in tqdm(range(len(ds)), desc="Checking audio decodability"):
        try:
            _ = ds[i]["audio"]["array"]
            valid.append(i)
        except Exception as e:
            corrupt.append((i, ds[i].get("id"), str(e).splitlines()[0][:120]))
    if corrupt:
        print(f"\n{len(corrupt)} corrupt clip(s) will be skipped:")
        for idx, cid, msg in corrupt:
            print(f"   idx={idx}  id={cid}  → {msg}")
    print(f"\nUsable utterances: {len(valid)}/{len(ds)}")
    return valid

valid_indices = collect_valid_indices(dataset)

Checking audio decodability: 100%|██████████| 428/428 [00:02<00:00, 206.60it/s]


1 corrupt clip(s) will be skipped:
   idx=274  id=1689  → decode_av_frame, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:1510, C

Usable utterances: 427/428


In [5]:
def prepare_audio(audio_array: np.ndarray, sr: int = 16_000) -> np.ndarray:
    """
    Ensure float32, mono, 16 kHz, range [-1, 1].
    FLEURS audio is already in this format, but this guard makes the
    notebook safe if you later swap datasets.
    """
    arr = np.asarray(audio_array, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=-1)          # downmix to mono
    if sr != 16_000:
        import librosa
        arr = librosa.resample(arr, orig_sr=sr, target_sr=16_000)
    # If values look like int16 range, normalise
    if np.abs(arr).max() > 1.5:
        arr = arr / 32768.0
    return np.clip(arr, -1.0, 1.0)

In [6]:
def transcribe_one(model, audio_array: np.ndarray, language: str) -> str:
    """
    audio_array : float32 NumPy array, 16 kHz, mono, [-1, 1].
    Returns raw transcription string.
    """
    audio = prepare_audio(audio_array)
    # OpenAI's transcribe accepts a NumPy array directly.
    # Explicit language avoids auto-detection overhead and wrong-language outputs.
    result = model.transcribe(
        audio,
        language=language.split("_")[0],   # "ar_eg" -> "ar"
        task="transcribe",
        fp16=False,                        # CPU: no FP16
        verbose=False,
    )
    return result["text"].strip()

In [22]:
from whisper.model import Whisper, ModelDimensions

# Register once at module scope — required by torch>=2.6 with weights_only=True
torch.serialization.add_safe_globals([ModelDimensions])

def load_from_pt(size: str, device: str = "cpu") -> Whisper:
    pt = MODEL_DIR / f"whisper_{size}.pt"
    ckpt = torch.load(pt, map_location=device, weights_only=True)
    model = Whisper(ckpt["dims"])
    model.load_state_dict(ckpt["model_state_dict"])
    return model.to(device).eval()


import time
import numpy as np

def evaluate_model(model_size: str, language: str, ds, indices: list[int]):
    """
    Zero-shot evaluation with WER, CER and per-utterance latency.

    Returns:
        metrics       : dict of aggregate metrics
        hypotheses    : list[str] (normalised)
        references    : list[str] (normalised)
        latencies_ms  : list[float] per-utterance wall-clock latency
    """
    print(f"\nLoading {MODEL_DIR.name}/whisper_{model_size}.pt …")
    model = load_from_pt(model_size, DEVICE)

    hypotheses, references = [], []
    latencies_ms, audio_durs_s = [], []

    for i in tqdm(indices, desc=f"Evaluating whisper-{model_size}"):
        audio = ds[i]["audio"]["array"]
        ref   = ds[i]["transcription"]

        try:
            t0 = time.perf_counter()
            hyp = transcribe_one(model, audio, language)
            t1 = time.perf_counter()
        except Exception as e:
            print(f"\nTranscription failed at idx={i} ({ds[i].get('id')}): {e}")
            continue

        latencies_ms.append((t1 - t0) * 1000.0)
        audio_durs_s.append(len(audio) / 16_000.0)   # seconds of audio
        hypotheses.append(normalizer(hyp))
        references.append(normalizer(ref))

    lat = np.asarray(latencies_ms, dtype=np.float64)
    dur = np.asarray(audio_durs_s, dtype=np.float64)

    metrics = {
        # Quality
        "wer_percent":      jiwer.wer(references, hypotheses) * 100.0,
        "cer_percent":      jiwer.cer(references, hypotheses) * 100.0,
        "n_evaluated":      len(references),
        # Latency (wall-clock, excludes dataset audio decoding)
        "latency_total_s":  lat.sum() / 1000.0,
        "latency_mean_ms":  lat.mean(),
        "latency_median_ms": float(np.median(lat)),
        "latency_p95_ms":   float(np.percentile(lat, 95)),
        "latency_min_ms":   lat.min(),
        "latency_max_ms":   lat.max(),
        # Audio duration
        "audio_total_s":    dur.sum(),
        "audio_mean_s":     dur.mean(),
        # RTF = inference_time / audio_duration; < 1 means faster than real time
        "rtf_mean":         (lat / 1000.0 / dur).mean(),
        "rtf_overall":      lat.sum() / 1000.0 / dur.sum(),
    }

    return metrics, hypotheses, references, latencies_ms

In [23]:
results = {}

for size in MODEL_SIZES:
    metrics, hyps, refs, lats = evaluate_model(size, LANG, dataset, valid_indices)
    results[size] = {**metrics, "hypotheses": hyps, "references": refs,
                     "latencies_ms": lats}

    print(f"\nwhisper-{size}")
    print(f"  WER           : {metrics['wer_percent']:6.2f} %")
    print(f"  CER           : {metrics['cer_percent']:6.2f} %")
    print(f"  Evaluated     : {metrics['n_evaluated']} utterances "
          f"({metrics['audio_total_s']:.1f}s of audio)")
    print(f"  Total time    : {metrics['latency_total_s']:6.1f} s")
    print(f"  Latency       : mean {metrics['latency_mean_ms']:7.1f} ms  |  "
          f"median {metrics['latency_median_ms']:7.1f} ms  |  "
          f"p95 {metrics['latency_p95_ms']:7.1f} ms")
    print(f"  Latency range : {metrics['latency_min_ms']:.0f} – "
          f"{metrics['latency_max_ms']:.0f} ms")
    print(f"  RTF           : mean {metrics['rtf_mean']:.3f}  |  "
          f"overall {metrics['rtf_overall']:.3f}")

    gc.collect()

# ------------------------------------------------------------------
# Comparison table
# ------------------------------------------------------------------
print("\n" + "=" * 78)
print(f"Zero-shot results on FLEURS {LANG}/{SPLIT}  "
      f"({len(valid_indices)} utterances)")
print("=" * 78)
print(f"{'Model':<16}{'WER %':>8}{'CER %':>8}{'Mean(ms)':>11}"
      f"{'p95(ms)':>10}{'RTF':>9}")
print("-" * 78)
for size, info in results.items():
    print(f"whisper-{size:<8}{info['wer_percent']:>8.2f}{info['cer_percent']:>8.2f}"
          f"{info['latency_mean_ms']:>11.1f}{info['latency_p95_ms']:>10.1f}"
          f"{info['rtf_mean']:>9.3f}")
print("=" * 78)


Loading baselines/whisper_tiny.pt …


Evaluating whisper-tiny: 100%|██████████| 427/427 [13:15<00:00,  1.86s/it]



whisper-tiny
  WER           :  67.72 %
  CER           :  26.84 %
  Evaluated     : 427 utterances (4683.1s of audio)
  Total time    :  792.4 s
  Latency       : mean  1855.7 ms  |  median  1427.8 ms  |  p95  3848.0 ms
  Latency range : 769 – 18599 ms
  RTF           : mean 0.180  |  overall 0.169

Loading baselines/whisper_small.pt …


Evaluating whisper-small: 100%|██████████| 427/427 [54:53<00:00,  7.71s/it]



whisper-small
  WER           :  31.92 %
  CER           :   9.15 %
  Evaluated     : 427 utterances (4683.1s of audio)
  Total time    : 3290.3 s
  Latency       : mean  7705.7 ms  |  median  7457.9 ms  |  p95 10816.0 ms
  Latency range : 4444 – 13647 ms
  RTF           : mean 0.752  |  overall 0.703

Zero-shot results on FLEURS ar_eg/test  (427 utterances)
Model              WER %   CER %   Mean(ms)   p95(ms)      RTF
------------------------------------------------------------------------------
whisper-tiny       67.72   26.84     1855.7    3848.0    0.180
whisper-small      31.92    9.15     7705.7   10816.0    0.752


In [24]:
import json
from pathlib import Path

out_path = Path(f"fleurs_{LANG}_{SPLIT}_openai_whisper_zeroshot.json")
payload = {
    "language": LANG,
    "split": SPLIT,
    "n_evaluated": len(valid_indices),
    "models": {
        size: {
            "model_size": size,
            # quality
            "wer_percent": info["wer_percent"],
            "cer_percent": info["cer_percent"],
            # latency
            "latency_mean_ms":   info["latency_mean_ms"],
            "latency_median_ms": info["latency_median_ms"],
            "latency_p95_ms":    info["latency_p95_ms"],
            "latency_total_s":   info["latency_total_s"],
            # throughput
            "audio_total_s": info["audio_total_s"],
            "rtf_mean":      info["rtf_mean"],
            "rtf_overall":   info["rtf_overall"],
        }
        for size, info in results.items()
    },
}
out_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False))
print(f"Saved → {out_path.resolve()}")

Saved → /home/matthiew/projects/Agent-Marketplace/whisper/src/fleurs_ar_eg_test_openai_whisper_zeroshot.json
